In [1]:
import pandas as pd

df = pd.read_csv("data/dataco_clean.csv", parse_dates=["order date (DateOrders)"])
df = df[df["Delivery Status"] != "Shipping canceled"]

df["order_month"] = df["order date (DateOrders)"].dt.month
df["order_weekday"] = df["order date (DateOrders)"].dt.dayofweek

features = ["Shipping Mode", "Market", "Order Region", "Category Name",
            "Department Name", "Customer Segment", "Type",
            "Order Item Quantity", "Order Item Discount Rate",
            "Order Item Product Price", "order_month", "order_weekday"]

X = df[features]
y = df["Late_delivery_risk"]

print(X.shape, y.mean().round(3))

(172765, 12) 0.573


In [2]:
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

cutoff = df["order date (DateOrders)"].quantile(0.8)
train = df["order date (DateOrders)"] < cutoff

X_train, X_test = X[train], X[~train]
y_train, y_test = y[train], y[~train]

mode_rate = y_train.groupby(X_train["Shipping Mode"]).mean()
base_score = X_test["Shipping Mode"].map(mode_rate)
base_pred = (base_score > 0.5).astype(int)

print("Cutoff date:", cutoff.date())
print("Always-guess-late accuracy:", round(y_test.mean(), 3))
print("Baseline accuracy:", round(accuracy_score(y_test, base_pred), 3))
print("Baseline F1:", round(f1_score(y_test, base_pred), 3))
print("Baseline ROC-AUC:", round(roc_auc_score(y_test, base_score), 3))

Cutoff date: 2017-04-22
Always-guess-late accuracy: 0.573
Baseline accuracy: 0.696
Baseline F1: 0.67
Baseline ROC-AUC: 0.744


In [3]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import HistGradientBoostingClassifier

cat_cols = ["Shipping Mode", "Market", "Order Region", "Category Name",
            "Department Name", "Customer Segment", "Type"]

prep = ColumnTransformer(
    [("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols)],
    remainder="passthrough")

model = Pipeline([("prep", prep),
                  ("clf", HistGradientBoostingClassifier(random_state=42))])

model.fit(X_train, y_train)

score = model.predict_proba(X_test)[:, 1]
pred = (score > 0.5).astype(int)

print("Model accuracy:", round(accuracy_score(y_test, pred), 3))
print("Model F1:", round(f1_score(y_test, pred), 3))
print("Model ROC-AUC:", round(roc_auc_score(y_test, score), 3))

Model accuracy: 0.695
Model F1: 0.674
Model ROC-AUC: 0.744
